# Phase 2b — DSPy Optimization + QLoRA Fine-Tuning (Tier 1, N=10)
Compares against the baseline (93.8% on Tier 1) from Phase 2.

**Before running:** Runtime -> Change runtime type -> T4 GPU. Do this BEFORE running any cell below.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: the cell above must show a Tesla T4 GPU table.** If it says `nvidia-smi: command not found`, go to Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Restart session, and re-run from the top. Do not continue past this point without a confirmed GPU.

## 1. Clone repo (safe to re-run any time)

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pwd

`!pwd` must show exactly `/content/agentic-prompt-vs-finetune` (no repeated folder name). It always will now — this cell wipes any leftover folder before cloning, so it can't nest no matter how many times you run it.

In [ ]:
!grep -A2 "MIPROv2(metric" dspy_optimize.py

This must show `num_threads=1` on the `dspy.MIPROv2(metric=...)` line, right after cloning, before installing anything else. This is now the ONLY clone cell and ONLY verify cell in the notebook — no more duplicated checks further down to get out of sync.

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets dspy-ai

In [ ]:
from huggingface_hub import login
login()

## 3. Sanity-check the training data generator (no GPU needed)

In [ ]:
!python envs/training_data.py

Check: pool size ~138, leakage check = 0. If leakage isn't 0, stop and flag it before continuing.

## 4. DSPy optimization (N=10)

In [ ]:
import sys, json
sys.path.insert(0, ".")
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from envs.training_data import sample_tier1_training
from tasks.tier1 import TIER1_HELDOUT
import dspy_optimize

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model + DSPy LM wrapper ready.")

In [ ]:
train_10 = sample_tier1_training(10)
print(f"Training on {len(train_10)} examples.")

optimized_program = dspy_optimize.optimize(lm, train_10)
print("DSPy optimization complete.")

If this cell errors, paste the FULL traceback. If it's a TypeError about kwargs, run `help(dspy.MIPROv2.__init__)` and `help(dspy.MIPROv2.compile)` in a new cell (import dspy first) and paste both before we adjust further. If it's a CUDA OutOfMemoryError, paste the traceback too — do not just retry blindly, since fragmentation can persist within a session.

In [ ]:
dspy_results = dspy_optimize.evaluate_program(optimized_program, TIER1_HELDOUT)
for r in dspy_results:
    print(f"[{'PASS' if r['grade']['success'] else 'FAIL'}] {r['id']} — {r['grade'].get('failure_type')}")

dspy_success_rate = sum(r["grade"]["success"] for r in dspy_results) / len(dspy_results)
print(f"\nDSPy-optimized (N=10) held-out success rate: {dspy_success_rate:.1%}")

with open("results/tier1_dspy_n10_results.json", "w") as f:
    json.dump(dspy_results, f, indent=2)

## 5. Free GPU memory before fine-tuning

In [ ]:
import gc, torch
del model, lm, optimized_program
gc.collect()
torch.cuda.empty_cache()
print("Memory freed.")

## 6. QLoRA fine-tuning (N=10)
This loads its own fresh copy of the model, trains a LoRA adapter, and saves it to `adapters/tier1_n10/`.

In [ ]:
!python qlora_finetune.py --n 10

Watch the training loss printed each step — it should trend downward. If it's flat or NaN, paste the output before continuing.

## 7. Evaluate the fine-tuned adapter on the same held-out set

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from envs.agent_harness import run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from grader import grade_task

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=bnb_config, device_map="auto")
ft_model = PeftModel.from_pretrained(base_model, "adapters/tier1_n10")
ft_tok = AutoTokenizer.from_pretrained("adapters/tier1_n10")
print("Fine-tuned model loaded.")

In [ ]:
qlora_results = []
for task in TIER1_HELDOUT:
    tool_calls, final_text = run_agent(ft_model, ft_tok, task["prompt"], TOOL_SCHEMAS, call_tool)
    grade = grade_task(task, tier=1, tool_calls=tool_calls, final_text=final_text)
    qlora_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

qlora_success_rate = sum(r["grade"]["success"] for r in qlora_results) / len(qlora_results)
print(f"\nQLoRA fine-tuned (N=10) held-out success rate: {qlora_success_rate:.1%}")

with open("results/tier1_qlora_n10_results.json", "w") as f:
    json.dump(qlora_results, f, indent=2)

## 8. Compare all three conditions

In [ ]:
print("Tier 1, N=10 data regime, evaluated on held-out set:")
print(f"  DSPy-optimized:   {dspy_success_rate:.1%}")
print(f"  QLoRA fine-tuned: {qlora_success_rate:.1%}")

Note: your Phase 2 baseline (93.8%) was measured on TIER1_TASKS, not TIER1_HELDOUT. For a fair 3-way comparison, run the baseline harness against TIER1_HELDOUT too (swap TIER1_TASKS for TIER1_HELDOUT in the Phase 2 notebook's Section 6 and re-run).

## 9. Download everything to push

In [ ]:
from google.colab import files
files.download("results/tier1_dspy_n10_results.json")
files.download("results/tier1_qlora_n10_results.json")
files.download("adapters/tier1_n10/training_examples.json")

Move the 3 downloaded files into `results/` on your laptop (rename `training_examples.json` to `training_examples_n10.json`), then:
```bash
git add results/tier1_dspy_n10_results.json results/tier1_qlora_n10_results.json results/training_examples_n10.json
git commit -m "Phase 2b: DSPy + QLoRA results for Tier 1, N=10"
git push
```